# Day 02: The Error-State Extended Kalman Filter (ES-EKF) - Dead-Reckoning & Sensor Fusion
**State Estimation and Localization for Self-Driving Cars**

### 🎯 Learning Objectives:
1. Master the **Error-State Formulation**: Decomposing state into a large nonlinear *nominal state* $\mathbf{x}_{\text{nom}}$ and small linear *error state* $\delta\mathbf{x}$.
2. Understand why linearizing about an error state near $\mathbf{0}$ virtually eliminates higher-order truncation errors and singularity hazards.
3. Implement the generic multi-dimensional `ErrorStateKalmanFilter` class matching rigorous academic standards.
4. Understand continuous white noise spectral density scaling for process noise $\mathbf{Q}_k \approx \mathbf{Q}_c \Delta t$.
5. Implement manifold error injection $(\hat{\mathbf{x}}_{\text{nom}} \leftarrow \check{\mathbf{x}}_{\text{nom}} \oplus \delta\hat{\mathbf{x}})$, error reset $(\delta\hat{\mathbf{x}} \leftarrow \mathbf{0})$, and Joseph form covariance reset.
6. Simulate high-rate dead-reckoning with periodic GNSS outages and observe 3-sigma uncertainty growth and recovery via Plotly.

---
## 1. Environment Setup

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2

np.random.seed(42)
print('Environment ready: NumPy, SciPy, and Plotly loaded.')

---
## 2. Mathematical Foundation: Nominal vs. Error State Formulation

In strapdown navigation and robotics, we decouple state estimation:

$$\mathbf{x} = \mathbf{x}_{\text{nom}} \oplus \delta\mathbf{x}$$

1. **Nominal State $\mathbf{x}_{\text{nom}}$**: High-rate deterministic kinematic integration of nonlinear motion (without noise).
2. **Error State $\delta\mathbf{x} \sim \mathcal{N}(\mathbf{0}, \mathbf{P})$**: Small perturbation error. Because true error stays close to zero, 1st-order Taylor approximations are exceptionally accurate.

### 🔁 Discrete ES-EKF Recursive Algorithm:

| Step | Mathematical Formula | Academic Definition |
| :--- | :--- | :--- |
| **1. Nominal Propagation** | $\check{\mathbf{x}}_{\text{nom}} = \mathbf{f}_{\text{nom}}(\hat{\mathbf{x}}_{\text{nom}}, \mathbf{u})$ | High-rate nonlinear kinematic dead-reckoning integration |
| **2. Covariance Propagation** | $\check{\mathbf{P}} = \mathbf{F}_{\delta}\hat{\mathbf{P}}\mathbf{F}_{\delta}^T + \mathbf{L}\mathbf{Q}\mathbf{L}^T$ | Error-state uncertainty expansion |
| **3. Innovation Residual** | $\mathbf{\nu} = \mathbf{y} - \mathbf{h}(\check{\mathbf{x}}_{\text{nom}})$ | Discrepancy between measurement and nominal state observation |
| **4. Innovation Covariance** | $\mathbf{S} = \mathbf{H}\check{\mathbf{P}}\mathbf{H}^T + \mathbf{M}\mathbf{R}\mathbf{M}^T$ | Total error uncertainty projected in measurement space |
| **5. Error Kalman Gain** | $\mathbf{K} = \check{\mathbf{P}}\mathbf{H}^T \mathbf{S}^{-1}$ | Optimal error-state Kalman gain |
| **6. Error Correction** | $\delta\hat{\mathbf{x}} = \mathbf{K}\mathbf{\nu}$ | Optimal estimate of the perturbation error |
| **7. State Injection** | $\hat{\mathbf{x}}_{\text{nom}} = \check{\mathbf{x}}_{\text{nom}} \oplus \delta\hat{\mathbf{x}}$ | Inject estimated error into nominal state (with manifold/angle wrapping) |
| **8. Error Reset** | $\delta\hat{\mathbf{x}} \leftarrow \mathbf{0}$ | Reset error mean to zero after injection |
| **9. Covariance Reset** | $\hat{\mathbf{P}} = (\mathbf{I} - \mathbf{K}\mathbf{H})\check{\mathbf{P}}(\mathbf{I} - \mathbf{K}\mathbf{H})^T + \mathbf{K}\mathbf{R}_{\text{eff}}\mathbf{K}^T$ | Joseph form error covariance reset |

---
## 3. Implementing the `ErrorStateKalmanFilter` Class

### 📝 Exercise 1: Implement the Generic ES-EKF Class

In [ ]:
def wrap_angle(angle):
    """Wraps angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class ErrorStateKalmanFilter:
    """Generic Multi-Dimensional Error-State Extended Kalman Filter (ES-EKF)."""
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray):
        self.x_nom = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.n = self.x_nom.shape[0]
        
        self.latest_innovation = None
        self.latest_innovation_cov = None
        self.latest_gain = None

    def predict(self, f_nom_func, F_jac: np.ndarray, Q: np.ndarray, 
                u: np.ndarray = None, L_jac: np.ndarray = None):
        # -------------------------------------------------------------------------
        # TODO 1.1: Implement generic nominal propagation & error covariance propagation
        # 1. Propagate nominal state: check_x_nom = f_nom(hat_x_nom, u)
        # 2. Propagate covariance: check_P = F_delta * hat_P * F_delta^T + L * Q * L^T
        # -------------------------------------------------------------------------
        pass  # <-- YOUR CODE HERE

    propagate = predict

    def update(self, y: np.ndarray, h_func, H_jac: np.ndarray, R: np.ndarray, 
               M_jac: np.ndarray = None, inject_func = None, angle_indices: list = None):
        # -------------------------------------------------------------------------
        # TODO 1.2: Implement generic measurement update, injection, and error reset
        # 1. Innovation: nu = y - h(check_x_nom)
        # 2. Innovation Covariance: S = H * check_P * H^T + M * R * M^T
        # 3. Kalman Gain: K = check_P * H^T * inv(S)
        # 4. Error Correction: delta_x = K * nu
        # 5. Injection: hat_x_nom = inject_func(check_x_nom, delta_x) (or + if None)
        # 6. Reset & Covariance Update: Joseph form (I - K*H) * check_P * (I - K*H)^T + K*R_eff*K^T
        # -------------------------------------------------------------------------
        pass  # <-- YOUR CODE HERE


In [ ]:
def wrap_angle(angle):
    """Wraps angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class ErrorStateKalmanFilter:
    """Generic Multi-Dimensional Error-State Extended Kalman Filter (ES-EKF).
    
    Maintains nominal kinematic state x_nom and error-state covariance P.
    Provides generic predict (propagate) and update methods completely decoupled
    from specific physical sensors or vehicle dimensions.
    """
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray):
        self.x_nom = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.n = self.x_nom.shape[0]
        
        self.latest_innovation = None
        self.latest_innovation_cov = None
        self.latest_gain = None

    def predict(self, f_nom_func, F_jac: np.ndarray, Q: np.ndarray, 
                u: np.ndarray = None, L_jac: np.ndarray = None):
        """Executes High-Rate Nominal State Propagation & Error Covariance Propagation:
            check_x_nom = f_nom(hat_x_nom, u)
            check_P = F_delta * hat_P * F_delta^T + L * Q * L^T
        """
        if u is not None:
            self.x_nom = f_nom_func(self.x_nom, u).reshape(-1, 1)
        else:
            self.x_nom = f_nom_func(self.x_nom).reshape(-1, 1)
            
        if L_jac is None:
            L_jac = np.eye(self.n)
            
        self.P = F_jac @ self.P @ F_jac.T + L_jac @ Q @ L_jac.T
        return self.x_nom.copy(), self.P.copy()

    # Alias matching strapdown dead-reckoning terminology
    propagate = predict

    def update(self, y: np.ndarray, h_func, H_jac: np.ndarray, R: np.ndarray, 
               M_jac: np.ndarray = None, inject_func = None, angle_indices: list = None):
        """Executes Generic Measurement Update, Nominal Injection, and Error Reset:
            1. Innovation:            nu = y - h(check_x_nom)
            2. Innovation Covariance: S  = H * check_P * H^T + M * R * M^T
            3. Error Kalman Gain:     K  = check_P * H^T * inv(S)
            4. Error Correction:      delta_x = K * nu
            5. State Injection:       hat_x_nom = check_x_nom (oplus) delta_x
            6. Error Reset:           delta_x <- 0
            7. Covariance Reset:      hat_P = (I - K*H) * check_P * (I - K*H)^T + K * R_eff * K^T
        """
        y_vec = np.asarray(y, dtype=np.float64).reshape(-1, 1)
        m = y_vec.shape[0]
        if M_jac is None:
            M_jac = np.eye(m)
            
        # 1. Innovation residual
        y_pred = h_func(self.x_nom).reshape(-1, 1)
        nu = y_vec - y_pred
        if angle_indices is not None:
            for idx in angle_indices:
                nu[idx, 0] = wrap_angle(nu[idx, 0])
                
        # 2. Innovation Covariance
        R_eff = M_jac @ R @ M_jac.T
        S = H_jac @ self.P @ H_jac.T + R_eff
        
        # 3. Optimal Error Kalman Gain
        K = self.P @ H_jac.T @ np.linalg.inv(S)
        
        # 4. Error State Estimate
        delta_x = K @ nu
        
        # 5. Nominal State Injection (hat_x_nom = check_x_nom (oplus) delta_x)
        if inject_func is not None:
            self.x_nom = inject_func(self.x_nom, delta_x).reshape(-1, 1)
        else:
            self.x_nom = (self.x_nom + delta_x).reshape(-1, 1)
            
        # 6 & 7. Covariance Reset (Joseph form for numerical stability)
        I_KH = np.eye(self.n) - K @ H_jac
        self.P = I_KH @ self.P @ I_KH.T + K @ R_eff @ K.T
        self.P = 0.5 * (self.P + self.P.T)
        
        self.latest_innovation = nu
        self.latest_innovation_cov = S
        self.latest_gain = K
        
        return self.x_nom.copy(), self.P.copy()

print('Generic ErrorStateKalmanFilter class compiled successfully.')

In [ ]:
def nominal_kinematics(x, u, dt):
    """High-rate nominal kinematic dead-reckoning integration f_nom(x, u)."""
    # TODO 2.1: Implement nominal kinematic integration
    pass

def get_error_F_jacobian(x, u, dt):
    """Computes 4x4 error-state transition Jacobian F_delta."""
    # TODO 2.2: Implement error-state Jacobian
    pass

def position_measurement_model(x):
    """Position measurement model h(x) = [px, py]^T."""
    # TODO 2.3: Implement position measurement model
    pass

def get_position_H_jacobian():
    """Computes 2x4 measurement Jacobian H = dh/d(delta_x)."""
    # TODO 2.4: Implement position measurement Jacobian
    pass

def inject_vehicle_state(x_nom, delta_x):
    """Manifold error injection x_nom = x_nom (oplus) delta_x."""
    # TODO 2.5: Implement manifold state injection with heading angle wrapping
    pass


In [ ]:
def nominal_kinematics(x, u, dt):
    """High-rate nominal kinematic dead-reckoning integration f_nom(x, u)."""
    px, py, v, theta = x.flatten()
    a, omega = u.flatten()
    px_next = px + v * np.cos(theta) * dt
    py_next = py + v * np.sin(theta) * dt
    v_next = v + a * dt
    theta_next = wrap_angle(theta + omega * dt)
    return np.array([px_next, py_next, v_next, theta_next]).reshape(-1, 1)

def get_error_F_jacobian(x, u, dt):
    """Computes 4x4 error-state transition Jacobian F_delta."""
    px, py, v, theta = x.flatten()
    return np.array([
        [1.0, 0.0, np.cos(theta) * dt, -v * np.sin(theta) * dt],
        [0.0, 1.0, np.sin(theta) * dt,  v * np.cos(theta) * dt],
        [0.0, 0.0, 1.0,                 0.0],
        [0.0, 0.0, 0.0,                 1.0]
    ])

def position_measurement_model(x):
    """Position measurement model h(x) = [px, py]^T."""
    return x[:2, :].reshape(-1, 1)

def get_position_H_jacobian():
    """Computes 2x4 measurement Jacobian H = dh/d(delta_x)."""
    return np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0]
    ])

def inject_vehicle_state(x_nom, delta_x):
    """Manifold error injection x_nom = x_nom (oplus) delta_x."""
    x_inj = x_nom + delta_x
    x_inj[3, 0] = wrap_angle(x_inj[3, 0])
    return x_inj

print('Physical vehicle dynamics and GNSS models compiled.')

---
## 5. Physical Formulation of IMU Process Noise $\mathbf{Q}_{\text{imu}}$ & GNSS Noise $\mathbf{R}_{\text{gps}}$

### 🔹 IMU Process Noise $\mathbf{Q}_{\text{imu}}$ (100 Hz, $\Delta t = 0.01\text{ s}$):
In strapdown navigation, IMU sensor noise is modeled as continuous white noise spectral density, scaled by step time $\Delta t$:
$$\mathbf{Q}_k \approx \mathbf{Q}_{\text{continuous}} \cdot \Delta t_{\text{imu}}$$
* $\sigma_v = 0.05\text{ m/s}/\sqrt{\text{s}}$: Accelerometer Velocity Random Walk (VRW).
* $\sigma_\theta = 0.01\text{ rad/s}/\sqrt{\text{s}}$: Gyroscope Angular Random Walk (ARW).
$$\mathbf{Q}_{\text{imu}} = \operatorname{diag}(0.01^2, 0.01^2, 0.05^2, 0.01^2) \times 0.01$$

### 🔹 GNSS Measurement Noise $\mathbf{R}_{\text{gps}}$ (10 Hz):
* $\sigma_{p_x} = 1.5\text{ m}, \sigma_{p_y} = 1.5\text{ m}$: Standard civilian single-frequency GPS pseudorange precision.
$$\mathbf{R}_{\text{gps}} = \operatorname{diag}(1.5^2, 1.5^2)$$

---
## 6. Simulation: Dead-Reckoning with 10s GNSS Outage

In [ ]:
dt_imu = 0.01   # 100 Hz IMU
T_total = 50.0
N_steps = int(T_total / dt_imu)
time = np.linspace(0, T_total, N_steps)

Q_imu = np.diag([0.01**2, 0.01**2, 0.05**2, 0.01**2]) * dt_imu
R_gps = np.diag([1.5**2, 1.5**2])

x_true = np.array([0.0, 0.0, 10.0, 0.0]).reshape(4, 1)
x_true_all = np.zeros((4, N_steps))

u_cmds = []
for k in range(N_steps):
    t = time[k]
    a = 0.2 * np.sin(0.1 * t)
    omega = 0.05 * np.cos(0.05 * t)
    u_cmds.append(np.array([a, omega]))

# Initialize Generic ES-EKF
x0_est = np.array([0.0, 0.0, 10.0, 0.0]).reshape(4, 1)
P0_est = np.diag([1.0**2, 1.0**2, 1.0**2, np.deg2rad(5.0)**2])
esekf = ErrorStateKalmanFilter(x0_est, P0_est)

x_est_all = np.zeros((4, N_steps))
P_diag_all = np.zeros((4, N_steps))
gps_meas_history = []

for k in range(N_steps):
    t = time[k]
    u = u_cmds[k].reshape(2, 1)
    w = np.random.multivariate_normal(np.zeros(4), Q_imu).reshape(4, 1)
    x_true = nominal_kinematics(x_true, u, dt_imu) + w
    x_true[3, 0] = wrap_angle(x_true[3, 0])
    x_true_all[:, k] = x_true.flatten()

    # 1. High-rate IMU dead-reckoning propagation (100 Hz)
    noisy_u = u + np.random.normal(0, [0.05, 0.01]).reshape(2, 1)
    F = get_error_F_jacobian(esekf.x_nom, noisy_u, dt_imu)
    esekf.predict(lambda x, u_in: nominal_kinematics(x, u_in, dt_imu), F, Q_imu, u=noisy_u)

    # 2. Low-rate GNSS measurement update (10 Hz) with 10s Outage Window [20s, 30s]
    is_gps_step = (k % 10 == 0)
    is_outage = (20.0 <= t <= 30.0)
    if is_gps_step and not is_outage:
        v_gps = np.random.multivariate_normal(np.zeros(2), R_gps).reshape(2, 1)
        y_gps = position_measurement_model(x_true) + v_gps
        H_pos = get_position_H_jacobian()
        esekf.update(y_gps, position_measurement_model, H_pos, R_gps, inject_func=inject_vehicle_state)
        gps_meas_history.append((t, y_gps[0, 0], y_gps[1, 0]))

    x_est_all[:, k] = esekf.x_nom.flatten()
    P_diag_all[:, k] = np.diag(esekf.P)

rmse_pos = np.sqrt(np.mean((x_true_all[:2, :] - x_est_all[:2, :])**2))
print(f'ES-EKF simulation completed. Position RMSE: {rmse_pos:.3f} m')

---
## 7. Plotly Interactive Outage Diagnostics

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '2D Trajectory Fusion (GNSS + IMU)',
        'Position Error vs. 3-Sigma (Outage Highlighted)',
        'Velocity Error vs. 3-Sigma Bound',
        'Heading Error vs. 3-Sigma Bound'
    )
)

gps_arr = np.array(gps_meas_history)
fig.add_trace(go.Scatter(x=x_true_all[0, :], y=x_true_all[1, :], mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=gps_arr[:, 1], y=gps_arr[:, 2], mode='markers', name='GNSS Fixes (10 Hz)', marker=dict(color='red', size=4, opacity=0.6)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_est_all[0, :], y=x_est_all[1, :], mode='lines', name='ES-EKF Estimate', line=dict(color='blue', width=2, dash='dash')), row=1, col=1)

pos_err = np.sqrt((x_true_all[0, :] - x_est_all[0, :])**2 + (x_true_all[1, :] - x_est_all[1, :])**2)
sigma_pos = 3.0 * np.sqrt(P_diag_all[0, :] + P_diag_all[1, :])
fig.add_trace(go.Scatter(x=time, y=pos_err, mode='lines', name='Position Error [m]', line=dict(color='blue', width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=time, y=sigma_pos, mode='lines', name='3-Sigma Bound [m]', line=dict(color='red', dash='dot', width=1.5)), row=1, col=2)

v_err = np.abs(x_true_all[2, :] - x_est_all[2, :])
sigma_v = 3.0 * np.sqrt(P_diag_all[2, :])
fig.add_trace(go.Scatter(x=time, y=v_err, mode='lines', name='Speed Error [m/s]', line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=sigma_v, mode='lines', name='3-Sigma Bound [m/s]', line=dict(color='red', dash='dot', width=1.5)), row=2, col=1)

th_err = np.abs(wrap_angle(x_true_all[3, :] - x_est_all[3, :]))
sigma_th = 3.0 * np.sqrt(P_diag_all[3, :])
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(th_err), mode='lines', name='Heading Error [deg]', line=dict(color='magenta', width=1.5)), row=2, col=2)
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(sigma_th), mode='lines', name='3-Sigma Bound [deg]', line=dict(color='red', dash='dot', width=1.5)), row=2, col=2)

# Add shaded GNSS outage box on Position Error
fig.add_vrect(x0=20.0, x1=30.0, fillcolor='yellow', opacity=0.25, line_width=0, annotation_text='GNSS Outage Window', annotation_position='top left', row=1, col=2)

fig.update_layout(title_text='Day 2 ES-EKF: Dead-Reckoning & GNSS/INS Fusion', template='plotly_white', height=750, width=1050)
fig.show()